In [2]:
import cvxpy as cp
import numpy as np
from itertools import product
from itertools import combinations
import time

In [3]:
def cal_pow(R, beta, h_val):
    sort_arr = []
    for j in range(len(beta)):
        sort_arr.append((beta[j]/h_val[j], j))
    sorted_list = [y[1] for y in sorted(sort_arr, key=lambda x: x[0])]
    pow_vals = np.zeros(len(beta))
    for i in range(len(beta)):
        idx = sorted_list[i]
        sum_i_M = np.sum([R[k] for k in range(idx, len(beta))])
        sum_ip1_M = np.sum([R[k] for k in range(min(idx+1, len(beta)), len(beta))])
        pow = (2**(sum_i_M) - 2**(sum_ip1_M))/h_val[idx]
        pow_vals[idx] = pow
    return pow_vals
#####################################################3
n_user = 2
h_bad =  0.1
h_good = 1
pr_h_bad = 0.5
pr_h_good = (1- pr_h_bad)
r_max = 1
h_vec_u1 = np.array([h_bad,  h_good]).tolist()
permutations_list = []
permutations_list.extend(list(product(h_vec_u1, repeat=n_user )))
combined_h_vec = []
for permutation in permutations_list:
    arr = []
    for zz in permutation:
        arr.append(zz)
    combined_h_vec.append(arr)
p_h_u1 = np.array([pr_h_bad, pr_h_good])
permutations_list = []
permutations_list.extend(list(product(p_h_u1, repeat=n_user )))
pr_h = []
for permutation in permutations_list:
    arr = []
    for k in permutation:
        arr.append(k)
    pr_h.append(np.prod(arr))
hp = pr_h
h = combined_h_vec
##############################################################3
v_k = n_user*[0.1]
alpha = 0.1
max_iter = 200
for itr in range(max_iter):
    print('itr', itr)
    B = v_k 
    rho = np.arange(0, r_max+1, 1)
    P_bar_U = [5, 5]
    W = list(product(rho, repeat=n_user))
    mu = cp.Variable((len(h), len(W) ), nonneg=True)
    p = []
    for x in range(n_user):
        temp  = 0
        for k in range(len(h)):
            for j in range(len(W)):
                if W[j][x] != 0:
                    temp+= mu[k][j] * hp[k]
        p.append(temp)
    P_u = []
    for x in range(n_user):
        temp = 0
        for k in range(len(h)):
            for j in range( len(W)):
                temp+= cal_pow(W[j], B, h[k])[x] * mu[k][j] *  hp[k]
        P_u.append(temp)
    obj_fun = 0
    for n in range(n_user):
        obj_fun += ((cp.inv_pos(p[n]) - 1) 
                    + B[n] * (P_u[n]-P_bar_U[n]))
    objective = cp.Minimize(obj_fun)
    constraints = []
    a_val = []
    for q in range(len(h)):
        a_val.append( cp.sum([ mu[q][j] for j in range(len(W))]))
    for a in a_val:
        constraints.append(a == 1)
    for i in range(len(h)):           
        for j in range(len(W)):
            constraints.append(0<= mu[i][j])
            constraints.append(mu[i][j] <= 1)
    prob = cp.Problem(objective, constraints)
    prob.solve()
    print('age', np.round(prob.value, 4))
    mu_opt = mu.value
    P_u_g = []
    for x in range(n_user):
        temp = 0
        for k in range(len(h)):
            for j in range( len(W)):
                temp+= cal_pow(W[j], B, h[k])[x] * mu_opt[k][j] *  hp[k]
        P_u_g.append(temp)
    print('pow', np.round(P_u_g, 3))
    subgradient = []
    for n in range(n_user):
        subgradient.append(P_u_g[n] - P_bar_U[n])
    eta_k = alpha / np.sqrt(itr + 1)
    for n in range(n_user):
        v_k[n] = max(0, v_k[n] + eta_k * subgradient[n])
    print('dual var', np.round( v_k, 3))
    print('-------------------------------------')


itr 0
age 0.3167
pow [3.5 3. ]
dual var [0 0]
-------------------------------------
itr 1
age 0.0
pow [11.   5.5]
dual var [0.424 0.035]
-------------------------------------
itr 2
age -0.6794
pow [1.  5.5]
dual var [0.193 0.064]
-------------------------------------
itr 3
age 0.163
pow [2.217 4.283]
dual var [0.054 0.028]
-------------------------------------
itr 4
age 0.3375
pow [10.214  5.5  ]
dual var [0.287 0.051]
-------------------------------------
itr 5
age -0.136
pow [1.385 5.115]
dual var [0.14  0.055]
-------------------------------------
itr 6
age 0.3091
pow [2.634 3.866]
dual var [0.05  0.013]
-------------------------------------
itr 7
age 0.3085
pow [10.927  5.5  ]
dual var [0.26 0.03]
-------------------------------------
itr 8
age -0.0393
pow [1.432 5.068]
dual var [0.141 0.033]
-------------------------------------
itr 9
age 0.3294
pow [2.404 4.096]
dual var [0.059 0.004]
-------------------------------------
itr 10
age 0.3481
pow [9.425 5.5  ]
dual var [0.192 0.019]

In [4]:
print('mu opt\n', np.round(mu_opt, 3))

mu opt
 [[0.    0.484 0.2   0.316]
 [0.    0.616 0.    0.384]
 [0.    0.    0.    1.   ]
 [0.    0.    0.    1.   ]]


# Single iter -fixed Beta

In [8]:
def cal_pow(R, beta, h_val):
    sort_arr = []
    for j in range(len(beta)):
        sort_arr.append((beta[j]/h_val[j], j))
    # print(beta, h_val, sort_arr)
    sorted_list = [y[1] for y in sorted(sort_arr, key=lambda x: x[0])]
    # print(sorted_list)
    pow_vals = np.zeros(len(beta))
    for i in range(len(beta)):
        idx = sorted_list[i]
        sum_i_M = np.sum([R[k] for k in range(idx, len(beta))])
        sum_ip1_M = np.sum([R[k] for k in range(min(idx+1, len(beta)), len(beta))])
        pow = (2**(sum_i_M) - 2**(sum_ip1_M))/h_val[idx]
        pow_vals[idx] = pow
    return pow_vals

n_user = 2
B = [1, 1]
h_bad =  0.1
h_good = 1
pr_h_bad = 0.5
pr_h_good = (1- pr_h_bad)
r_max = 1
h_vec_u1 = np.array([h_bad,  h_good]).tolist()
permutations_list = []
permutations_list.extend(list(product(h_vec_u1, repeat=n_user )))
combined_h_vec = []
for permutation in permutations_list:
    arr = []
    for zz in permutation:
        arr.append(zz)
    combined_h_vec.append(arr)
p_h_u1 = np.array([pr_h_bad, pr_h_good])
permutations_list = []
permutations_list.extend(list(product(p_h_u1, repeat=n_user )))
pr_h = []
for permutation in permutations_list:
    arr = []
    for k in permutation:
        arr.append(k)
    pr_h.append(np.prod(arr))
hp = pr_h
h = combined_h_vec
#####################################################
rho = np.arange(0, r_max+1, 1)
P_bar_U = [5, 5]
W = list(product(rho, repeat=n_user))
print(W)
mu = cp.Variable((len(h), len(W) ), nonneg=True)
p = []
for x in range(n_user):
    temp  = 0
    for k in range(len(h)):
        for j in range(len(W)):
            if W[j][x] != 0:
                temp+= mu[k][j] * hp[k]
    p.append(temp)
P_u = []
for x in range(n_user):
    temp = 0
    for k in range(len(h)):
        for j in range( len(W)):
            temp+= cal_pow(W[j], B, h[k])[x] * mu[k][j] *  hp[k]
    P_u.append(temp)
obj_fun = 0
for n in range(n_user):
    obj_fun += ( (cp.inv_pos(p[n]) - 1) )
objective = cp.Minimize(obj_fun)
constraints = []
for n in range(n_user):
    constraints.append(P_u[n] <= P_bar_U[n] )
a_val = []
for q in range(len(h)):
    a_val.append( cp.sum([ mu[q][j] for j in range(len(W))]))
for a in a_val:
    constraints.append(a == 1)
for i in range(len(h)):           
    for j in range(len(W)):
        constraints.append(0<= mu[i][j])
        constraints.append(mu[i][j] <= 1)
prob = cp.Problem(objective, constraints)
prob.solve()
print('age', np.round(prob.value, 4))
print('mu', np.round(mu.value, 3))
for n in range(n_user):
    print('Pow',n,':',  np.round(P_u[n].value, 3))
    print('p', n, ':', np.round(p[n].value, 3))

[(np.int64(0), np.int64(0)), (np.int64(0), np.int64(1)), (np.int64(1), np.int64(0)), (np.int64(1), np.int64(1))]
age 0.4319
mu [[0.    0.477 0.2   0.323]
 [0.    0.623 0.    0.377]
 [0.    0.    0.    1.   ]
 [0.    0.    0.    1.   ]]
Pow 0 : 5.0
p 0 : 0.725
Pow 1 : 5.0
p 1 : 0.95
